# Clothes segmentation: test on a new dataset the model has never seen

**Dataset:** Clothing Co-Parsing (CCP), [github.com/bearpaw/clothing-co-parsing](https://github.com/bearpaw/clothing-co-parsing), Apache-2.0 licence. 1,004 street-fashion photos (about 550x830) with pixel labels in 59 classes. It is downloaded here; nothing is trained on it.

**Three models are compared on the same photos:**

1. **Before training:** our SegFormer-B3 exactly as it was before we trained it: ImageNet encoder weights and an untrained decoder. It shows what our training added.
2. **A public model:** `mattmdjaga/segformer_b2_clothes` from Hugging Face, a SegFormer-B2 that someone else trained on ATR. It shows how we compare with what anyone can download.
3. **Our final model:** `final_segformer_b3_e30`, with flip averaging.

**Rules fixed before the result** (PLAN.md, section 41): CCP's 59 labels are mapped to 13 comparison classes (face, arms and legs become one "skin" class, because CCP has only "skin"); labels without a clear match (for example stockings, socks, jewellery) are ignored. CCP photos that are also in ATR, found by image hash, are removed, because ATR was partly built from older fashion datasets.

A **T4 GPU is enough**. About 15 minutes. Run the cells from top to bottom.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/Eyad-Elnakib1/Atelier.git"  # this repository; set to "" to use an uploaded clothseg_code.zip instead

DRIVE_ROOT = "/content/drive/MyDrive/clothseg"
CODE_DIR = "/content/clothes-segmentation"
CODE_VERSION = "19"
HEIGHT, WIDTH = 576, 384
SESSION_BUDGET_MIN = 600   # used by a helper in cell 2; nothing is trained here

ATR_CACHE = f"/content/data/atr_{HEIGHT}x{WIDTH}"   # its meta.csv holds the image hash of every ATR photo
OUR_CHECKPOINT = f"{DRIVE_ROOT}/runs/final_segformer_b3_e30/checkpoints/best.pt"
BEFORE_PATH = "/content/before_training.pt"
PUBLIC_MODEL = "mattmdjaga/segformer_b2_clothes"

## 2. Mount Drive and show the environment

In [ ]:
import os, platform, shutil, time
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
SESSION_START = time.time()

def minutes_left():
    """Minutes of the session budget that remain; passed to every training command."""
    return max(1, SESSION_BUDGET_MIN - (time.time() - SESSION_START) / 60)

import torch, transformers
print("python      ", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: change the runtime type to a GPU")
if torch.cuda.is_available():
    print("GPU memory  ", f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("CPU cores   ", os.cpu_count())
print("local disk  ", f"{shutil.disk_usage('/content').free / 1024**3:.0f} GB free")
print("Drive       ", f"{shutil.disk_usage(DRIVE_ROOT).free / 1024**3:.0f} GB free")
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > L4 GPU, then run again."

## 3. Get the code

In [ ]:
import subprocess, zipfile

def code_version(folder):
    path = os.path.join(folder, "VERSION")
    return open(path).read().strip() if os.path.exists(path) else "1"

os.chdir("/content")
if REPO_URL:
    if os.path.isdir(f"{CODE_DIR}/.git"):
        print(subprocess.run(["git", "-C", CODE_DIR, "pull"], capture_output=True, text=True).stdout)
    else:
        r = subprocess.run(["git", "clone", REPO_URL, CODE_DIR], capture_output=True, text=True)
        assert r.returncode == 0, f"git clone failed. Is the URL correct and the repository public?\n{r.stderr}"
elif os.path.isdir(f"{DRIVE_ROOT}/code") and code_version(f"{DRIVE_ROOT}/code") == CODE_VERSION:
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    shutil.copytree(f"{DRIVE_ROOT}/code", CODE_DIR)
    print(f"code version {CODE_VERSION} copied from Drive")
else:
    from google.colab import files
    if os.path.exists("clothseg_code.zip"):
        os.remove("clothseg_code.zip")   # otherwise Colab renames the new upload
    print(f"Drive has code version {code_version(f'{DRIVE_ROOT}/code')}, this notebook needs version {CODE_VERSION}.")
    print("Choose the NEW clothseg_code.zip from the project folder on your computer.")
    uploaded = files.upload()
    assert "clothseg_code.zip" in uploaded, f"Expected a file named clothseg_code.zip, got {list(uploaded)}."
    for folder in (CODE_DIR, f"{DRIVE_ROOT}/code"):   # the Drive copy is used by later sessions
        shutil.rmtree(folder, ignore_errors=True)
        with zipfile.ZipFile("clothseg_code.zip") as z:
            z.extractall(folder)
    print("code uploaded and saved to Drive")

assert code_version(CODE_DIR) == CODE_VERSION, (
    f"The code is version {code_version(CODE_DIR)} but this notebook needs version {CODE_VERSION}. "
    "You uploaded an old clothseg_code.zip. Run this cell again and choose the new one.")
for needed in ("scripts/train.py", "scripts/resume_test.py", "src/clothseg/engine.py", "requirements.txt"):
    assert os.path.exists(f"{CODE_DIR}/{needed}"), f"{needed} is missing from {CODE_DIR}. Check the upload or the repository."
os.chdir(CODE_DIR)
print("code ready in", os.getcwd(), "| version", code_version(CODE_DIR))

## 4. Install requirements and prepare ATR (needed only for the overlap check)

In [ ]:
!pip install -q -r requirements.txt
import segmentation_models_pytorch, timm, cv2
print("segmentation_models_pytorch", segmentation_models_pytorch.__version__, "| timm", timm.__version__, "| opencv", cv2.__version__)
!python scripts/prepare_data.py --drive-root "{DRIVE_ROOT}" --height {HEIGHT} --width {WIDTH}

## 5. Download CCP

In [ ]:
import os, subprocess

CCP_DIR = "/content/ccp"
if not os.path.isdir(f"{CCP_DIR}/annotations/pixel-level"):
    print("downloading the Clothing Co-Parsing dataset from GitHub (about 130 MB) ...", flush=True)
    r = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/bearpaw/clothing-co-parsing.git", CCP_DIR],
                       capture_output=True, text=True)
    assert r.returncode == 0, "Download failed. Send me this message: " + r.stderr[-2000:]
n_ann = len([f for f in os.listdir(f"{CCP_DIR}/annotations/pixel-level") if f.endswith(".mat")])
n_img = len([f for f in os.listdir(f"{CCP_DIR}/photos") if f.endswith(".jpg")])
print(f"CCP ready: {n_img} photos, {n_ann} with pixel labels (only these are used)")

## 6. Read the labels, map them, and remove photos that are also in ATR

In [ ]:
import glob, os, sys
import numpy as np
import pandas as pd
import scipy.io as sio
from PIL import Image

sys.path.insert(0, f"{CODE_DIR}/src")
from clothseg.data.preprocess import _dhash
from clothseg.inference import load_image

# The classes all models are compared on. CCP has one "skin" class, so face, arms and legs
# of the models are merged into "skin" before comparing.
EVAL = ["background", "hat", "hair", "sunglasses", "upper-clothes", "skirt", "pants", "dress", "belt", "shoes", "skin", "bag", "scarf"]
GARMENTS = ["hat", "sunglasses", "upper-clothes", "skirt", "pants", "dress", "belt", "shoes", "bag", "scarf"]
# CCP label -> comparison class. Mapping fixed before any result (PLAN.md section 41); labels
# without a clear match are ignored (left out of the scores), not counted as errors.
CCP_TO_EVAL = {
    "null": "background", "hat": "hat", "hair": "hair", "sunglasses": "sunglasses", "skin": "skin",
    "blazer": "upper-clothes", "blouse": "upper-clothes", "cape": "upper-clothes", "cardigan": "upper-clothes",
    "coat": "upper-clothes", "hoodie": "upper-clothes", "jacket": "upper-clothes", "shirt": "upper-clothes",
    "sweater": "upper-clothes", "sweatshirt": "upper-clothes", "t-shirt": "upper-clothes", "top": "upper-clothes",
    "vest": "upper-clothes",
    "skirt": "skirt", "dress": "dress", "belt": "belt", "scarf": "scarf", "bag": "bag", "purse": "bag",
    "jeans": "pants", "leggings": "pants", "pants": "pants", "shorts": "pants",
    "boots": "shoes", "clogs": "shoes", "flats": "shoes", "heels": "shoes", "loafers": "shoes", "pumps": "shoes",
    "sandals": "shoes", "shoes": "shoes", "sneakers": "shoes", "wedges": "shoes",
}
CCP_IGNORED = {"accessories", "bodysuit", "bra", "bracelet", "earrings", "glasses", "gloves", "intimate", "jumper",
               "necklace", "panties", "ring", "romper", "socks", "stockings", "suit", "swimwear", "tie", "tights",
               "wallet", "watch"}
OVERLAP_MAX_BITS = 4   # a CCP photo whose hash differs from an ATR photo's in at most 4 of 64 bits counts as the same photo

names = [str(a[0]) if a.size else "" for a in sio.loadmat(f"{CCP_DIR}/label_list.mat")["label_list"].ravel()]
unknown = [n for n in names if n not in CCP_TO_EVAL and n not in CCP_IGNORED]
assert not unknown, f"CCP labels without a rule: {unknown}"
ccp_lut = np.full(256, 255, np.uint8)
for i, n in enumerate(names):
    if n in CCP_TO_EVAL:
        ccp_lut[i] = EVAL.index(CCP_TO_EVAL[n])

# Hashes of every ATR photo, computed the same way during preprocessing (on the original photo).
meta = pd.read_csv(f"{ATR_CACHE}/meta.csv", dtype={"dhash": str})
atr_hash = np.array([int(h, 16) for h in meta["dhash"]], dtype=np.uint64)

samples, skipped, ignored_px, total_px = [], [], 0, 0
for path in sorted(glob.glob(f"{CCP_DIR}/annotations/pixel-level/*.mat")):
    sid = os.path.basename(path)[:-4]
    raw = sio.loadmat(path)["groundtruth"]
    photo = f"{CCP_DIR}/photos/{sid}.jpg"
    if not os.path.exists(photo):
        skipped.append((sid, "no photo"))
        continue
    image = load_image(photo)
    if image.shape[:2] != raw.shape:
        skipped.append((sid, f"photo {image.shape[:2]} and labels {raw.shape} differ in size"))
        continue
    gt = ccp_lut[raw]
    ignored_px += int((gt == 255).sum())
    total_px += gt.size
    h = np.uint64(int(_dhash(Image.fromarray(image)), 16))
    bits = int(np.unpackbits((atr_hash ^ h).view(np.uint8)).reshape(-1, 64).sum(1).min())
    samples.append({"id": sid, "photo": photo, "gt": gt, "bits_to_nearest_atr": bits})

dist = np.array([s["bits_to_nearest_atr"] for s in samples])
clean = [s for s in samples if s["bits_to_nearest_atr"] > OVERLAP_MAX_BITS]
print(f"labelled photos read: {len(samples)}; skipped: {len(skipped)} {skipped[:3]}")
print(f"pixels with an ignored CCP label: {100 * ignored_px / max(1, total_px):.2f}%")
print(f"overlap with ATR (64-bit image hash): identical {int((dist == 0).sum())}, within {OVERLAP_MAX_BITS} bits {int((dist <= OVERLAP_MAX_BITS).sum())}, "
      f"within 8 bits {int((dist <= 8).sum())}")
print(f"photos used for the comparison (no overlap with ATR): {len(clean)}")
present = {c: sum(int((s["gt"] == k).any()) for s in clean) for k, c in enumerate(EVAL)}
print("photos containing each class:", ", ".join(f"{c} {n}" for c, n in present.items()))

## 7. Load the three models

In [ ]:
import time
import torch
import torch.nn.functional as F
from clothseg.config import TrainConfig
from clothseg.inference import Segmenter
from clothseg.models import build_model


def eval_lut(model_names):
    """Lookup table from a model's class ids to the comparison classes (left/right and body parts merged)."""
    lut = np.full(256, 255, np.uint8)
    for i, n in enumerate(model_names):
        n = n.lower().replace("_", "-")
        n = "skin" if n in ("face", "arms", "legs", "left-arm", "right-arm", "left-leg", "right-leg") else n
        n = "shoes" if n in ("left-shoe", "right-shoe") else n
        assert n in EVAL, f"no comparison class for model class {n!r}"
        lut[i] = EVAL.index(n)
    return lut


# 1. Our final model.
ours = Segmenter(OUR_CHECKPOINT, flip_tta=True)
ours_lut = eval_lut(ours.names)

# 2. The same network before our training: ImageNet encoder weights, decoder not trained yet.
torch.manual_seed(0)
start = build_model(ours.cfg.model, ours.num_classes, pretrained=True)
torch.save({"model": start.state_dict(), "config": ours.cfg.to_dict(), "epoch": 0}, BEFORE_PATH)
before = Segmenter(BEFORE_PATH, flip_tta=True)
before_lut = eval_lut(before.names)
del start

# 3. A public clothes model from Hugging Face, also trained on ATR by someone else.
from transformers import AutoImageProcessor, SegformerForSemanticSegmentation
public_proc = AutoImageProcessor.from_pretrained(PUBLIC_MODEL)
public = SegformerForSemanticSegmentation.from_pretrained(PUBLIC_MODEL).to(ours.device).eval()
public_lut = eval_lut([public.config.id2label[i] for i in range(public.config.num_labels)])


@torch.no_grad()
def predict_public(image):
    inputs = public_proc(images=Image.fromarray(image), return_tensors="pt").to(ours.device)
    logits = public(**inputs).logits
    logits = F.interpolate(logits, size=image.shape[:2], mode="bilinear", align_corners=False)
    return logits.argmax(1)[0].to(torch.uint8).cpu().numpy()


MODELS = {
    "before training (ImageNet start)": lambda im: before_lut[before.predict(im)],
    "public model": lambda im: public_lut[predict_public(im)],
    "our final model": lambda im: ours_lut[ours.predict(im)],
}
print("models ready:", ", ".join(MODELS), f"| public model: {PUBLIC_MODEL}")

## 8. Score the three models on the same photos

In [ ]:
K = len(EVAL)
garment_idx = [EVAL.index(c) for c in GARMENTS]


def scores(cm):
    inter = np.diag(cm).astype(float)
    union = cm.sum(0) + cm.sum(1) - np.diag(cm)
    iou = np.where(union > 0, inter / np.maximum(union, 1), np.nan)
    g = np.zeros((2, 2))
    is_g = np.isin(np.arange(K), garment_idx)
    for a in (0, 1):
        for b in (0, 1):
            g[a, b] = cm[np.ix_(is_g == a, is_g == b)].sum()
    return {"garment mIoU": np.nanmean(iou[garment_idx]), "mIoU (13 classes)": np.nanmean(iou),
            "clothing vs rest IoU": g[1, 1] / max(1, g[1, 1] + g[0, 1] + g[1, 0]),
            "pixel accuracy": inter.sum() / max(1, cm.sum()), **{c: iou[k] for k, c in enumerate(EVAL) if k}}


results, cms, preds = {}, {}, {name: {} for name in MODELS}
show_ids = {clean[i]["id"] for i in np.random.default_rng(0).permutation(len(clean))[:6]}
for name, fn in MODELS.items():
    cm = np.zeros((K, K), np.int64)
    t0 = time.time()
    for s in clean:
        image = load_image(s["photo"])
        p = fn(image)
        valid = s["gt"] != 255
        cm += np.bincount(s["gt"][valid].astype(np.int64) * K + p[valid], minlength=K * K).reshape(K, K)
        if s["id"] in show_ids:
            preds[name][s["id"]] = p
    results[name] = {**scores(cm), "ms per photo": 1000 * (time.time() - t0) / len(clean)}
    cms[name] = cm
    print(f"{name}: garment mIoU {results[name]['garment mIoU']:.4f}", flush=True)

table = pd.DataFrame(results)
print()
print(f"CCP, {len(clean)} photos not in ATR, original size")
print(table.round(4).to_string())
os.makedirs(f"{DRIVE_ROOT}/reports", exist_ok=True)
table.to_csv(f"{DRIVE_ROOT}/reports/ccp_comparison.csv")
print("saved", f"{DRIVE_ROOT}/reports/ccp_comparison.csv")

## 9. Six example photos side by side

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

PAL = np.array([[0, 0, 0], [128, 0, 0], [255, 0, 0], [170, 0, 51], [255, 85, 0], [0, 128, 0], [0, 85, 85], [0, 0, 85],
                [85, 51, 0], [255, 255, 0], [85, 255, 170], [0, 119, 221], [52, 86, 128]], np.uint8)
PAL_IGN = np.vstack([PAL, np.full((256 - len(PAL), 3), 255, np.uint8)])   # ignored pixels shown white
ids = sorted(preds["our final model"])
cols = ["photo", "CCP labels"] + list(MODELS)
fig, axes = plt.subplots(len(ids), len(cols), figsize=(3 * len(cols), 4.4 * len(ids)), squeeze=False)
for r, sid in enumerate(ids):
    s = next(x for x in clean if x["id"] == sid)
    pics = [load_image(s["photo"]), PAL_IGN[s["gt"]]] + [PAL[preds[m][sid]] for m in MODELS]
    for c, (title, pic) in enumerate(zip(cols, pics)):
        axes[r, c].imshow(pic)
        axes[r, c].axis("off")
        if r == 0:
            axes[r, c].set_title(title, fontsize=11)
handles = [mpatches.Patch(color=PAL[k] / 255, label=n) for k, n in enumerate(EVAL)] + [mpatches.Patch(color="white", ec="gray", label="ignored")]
fig.legend(handles=handles, loc="lower center", ncol=7, fontsize=10, frameon=False)
fig.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig(f"{DRIVE_ROOT}/reports/ccp_examples.png", dpi=110, bbox_inches="tight")
plt.show()
print("saved", f"{DRIVE_ROOT}/reports/ccp_examples.png")

## What to send back

Everything cells 6 and 8 print, and the picture from cell 9.